<a href="https://colab.research.google.com/github/NatWhitt/AIDI-Assignment/blob/Second-day-of-work/AIDI_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Imports
import pandas as pd
import spacy
import re

#NLP model
nlp = spacy.load("en_core_web_sm")

# Load CSV
df = pd.read_csv('Data with Message.csv')
message = df['Message']

In [2]:
# import spacy

# nlp = spacy.load("en_core_web_sm")
# ruler = nlp.add_pipe("entity_ruler")
# patterns = [{"label": "ORG", "pattern": "MyCorp Inc."}]
# ruler.add_patterns(patterns)

# doc = nlp("MyCorp Inc. is a company in the U.S.")
# print([(ent.text, ent.label_) for ent in doc.ents])

In [3]:
#Custom Labels for Type
types = ("lateness", "school", "department", "head", "personal", "academic")
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = []
for t in types:
    patterns.append({"label":"DTYPE", "pattern":[{"LOWER":t}]})
ruler.add_patterns(patterns)


In [4]:
#Function to extract the entities from the message. Returning them into a pandas series.
def extract_details(text):
    if pd.isna(text):
        return pd.Series(["None", "None", "None","None","None"])

    doc = nlp(text)

    # Initialize variables
    student = "Not found"
    date = "Not found"
    time = "Not found"
    dtype = "Not found"
    reason = "Not found"

    reason_tokens = []



    #Extract known entities
    for ent in doc.ents:
      if ent.label_ == "PERSON" and student == "Not found":
            student = ent.text
      elif ent.label_ == "DATE" and date == "Not found":
            date = ent.text
      elif ent.label_ == "TIME" and time == "Not found":
            time = ent.text
      elif ent.label_ == "DTYPE" and dtype == "Not found":
            dtype = ent.text

    #Extract reason - this will need to either be changed to a sept list similar to detention type or changed so that it always follows a set pattern IE reason: or for:
    for token in doc:
      if token.text.lower() == "for" and token.is_sent_start and reason == 'Not found':
        reason = ""
        # for reason_token in doc[token.i + 1:]:
        #   reason_tokens.append(reason_token.text)

        # # reason_tokens = [token.text for reason_token in doc]
        # reason = " ".join(reason_tokens).strip()
        reason_tokens = [reason_token.text for reason_token in doc[token.i + 1:]]
        joinedreason = " ".join(reason_tokens).strip()

        #Accomidate for half-term
        reason = joinedreason.replace('half - term','half-term')

        #Remove spaces before punctuation at the end of sentances or commas
        reason = re.sub(r'\s([,.!?])', r'\1', reason)

        #Removes extra spaces around ()
        reason = re.sub(r'\(\s', '(', reason)
        reason = re.sub(r'\s\)', ')', reason)

        #Removes spaces around /
        reason = re.sub(r'\s/\s', '/', reason)


        #Removes extra space before ' in words
        reason = re.sub(r"\s'(\w)",r"'\1",reason)



        break
    return pd.Series([student, date, time, dtype, reason])

In [5]:
text = "School detention set for Donna Harris-Martin on 27-11-2025 at 16:00:00. For Donna has already received two flags for misbehaving in Maths this half term and is therefore issued with a detention. In addition, in today's lesson Donna rolled a ball across the floor and did not show any working in his exercise book for the first half of the lesson. We have had many discussions together about the need to show working in Maths, instead of just guessing the answer, so this needs to change."
print(extract_details(text))

0                                  Donna Harris-Martin
1                                           27-11-2025
2                                             16:00:00
3                                               School
4    Donna has already received two flags for misbe...
dtype: object


In [6]:
message.head()

,Message
0,Lateness detention set for Jessica-Susan Ande...
1,School detention set for Walter Page on 12-02-...
2,Department detention set for Michael-Lawrence ...
3,Head detention set for Sarah Green on 15-10-20...
4,Lateness detention set for Emily Vargas-Major...


In [7]:
# Apply the extract function to the 'message' column
df[['Student', 'Date', 'Time', 'Type', 'Reason']] = message.apply(extract_details)

# # Save the results
# df.to_csv('Processed.csv', index=False)
df.head()

,Expected_Student,Expected_Date,Expected_Time,Expected_Reason,Expected_Type,Message,Student,Date,Time,Type,Reason
0,Jessica-Susan Anderson,09-12-2025,12:20:00,Lateness to School and/or lessons.,Lateness,Lateness detention set for Jessica-Susan Ande...,Jessica,09-12-2025,12:20:00,Lateness,Lateness to School and/or lessons.
1,Walter Page,12-02-2026,16:00:00,Repeatedly unbuckling his seat belt on a schoo...,School,School detention set for Walter Page on 12-02-...,Walter Page,12-02-2026,16:00:00,School,Repeatedly unbuckling his seat belt on a schoo...
2,Michael-Lawrence Robinson,05-02-2026,13:00:00,Failure to complete Science Sparx prep. Please...,Department,Department detention set for Michael-Lawrence ...,Michael-Lawrence Robinson,05-02-2026,13:00:00,Department,Failure to complete Science Sparx prep. Please...
3,Sarah Green,15-10-2025,12:50:00,3x warnings accrued since the beginning of hal...,Head,Head detention set for Sarah Green on 15-10-20...,Sarah Green,15-10-2025,12:50:00,Head,3x warnings accrued since the beginning of hal...
4,Emily Vargas-Majors,02-12-2025,12:20:00,Lateness to School and/or lessons.,Lateness,Lateness detention set for Emily Vargas-Major...,Emily Vargas-Majors,Not found,12:20:00,Lateness,Lateness to School and/or lessons.


In [8]:
df['Student_check'] = df['Expected_Student'] == (df['Student'])
df['Date_check'] = df['Expected_Date'] == (df['Date'])
df['Time_check'] = df['Expected_Time'] == (df['Time'])
df['Type_check'] = df['Expected_Type'] == (df['Type'])
df['Reason_check'] = df['Expected_Reason'] == (df['Reason'])
df.to_csv('ProcessedWithTesting.csv', index=False)
df.head()

,Expected_Student,Expected_Date,Expected_Time,Expected_Reason,Expected_Type,Message,Student,Date,Time,Type,Reason,Student_check,Date_check,Time_check,Type_check,Reason_check
0,Jessica-Susan Anderson,09-12-2025,12:20:00,Lateness to School and/or lessons.,Lateness,Lateness detention set for Jessica-Susan Ande...,Jessica,09-12-2025,12:20:00,Lateness,Lateness to School and/or lessons.,False,True,True,False,True
1,Walter Page,12-02-2026,16:00:00,Repeatedly unbuckling his seat belt on a schoo...,School,School detention set for Walter Page on 12-02-...,Walter Page,12-02-2026,16:00:00,School,Repeatedly unbuckling his seat belt on a schoo...,True,True,True,True,True
2,Michael-Lawrence Robinson,05-02-2026,13:00:00,Failure to complete Science Sparx prep. Please...,Department,Department detention set for Michael-Lawrence ...,Michael-Lawrence Robinson,05-02-2026,13:00:00,Department,Failure to complete Science Sparx prep. Please...,True,True,True,True,True
3,Sarah Green,15-10-2025,12:50:00,3x warnings accrued since the beginning of hal...,Head,Head detention set for Sarah Green on 15-10-20...,Sarah Green,15-10-2025,12:50:00,Head,3x warnings accrued since the beginning of hal...,True,True,True,True,True
4,Emily Vargas-Majors,02-12-2025,12:20:00,Lateness to School and/or lessons.,Lateness,Lateness detention set for Emily Vargas-Major...,Emily Vargas-Majors,Not found,12:20:00,Lateness,Lateness to School and/or lessons.,True,False,True,False,True
